In [2]:
from huggingface_hub import notebook_login

notebook_login()

In [17]:
from nnsight import LanguageModel

llm = LanguageModel("meta-llama/Llama-3.1-8B", device_map="auto")
print(llm)


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
  

In [18]:
# let's set up a few-shot prompt to see if models can reason about factuality
PROMPT_TEMPLATE = """The city of Tokyo is in Japan. This statement is: TRUE
The city of Hanoi is in Poland. This statement is: FALSE
{statement} This statement is:"""

source_statement = "The city of Toronto is in Canada." # true
source_prompt = PROMPT_TEMPLATE.format(statement=source_statement)
base_statement = "The city of Chicago is in Canada." # false
base_prompt = PROMPT_TEMPLATE.format(statement=base_statement)

# this is a false statement
print(base_prompt)

The city of Tokyo is in Japan. This statement is: TRUE
The city of Hanoi is in Poland. This statement is: FALSE
The city of Chicago is in Canada. This statement is:


In [19]:
import torch

model = llm
# does the model know that Chicago isn't in Canada?
with torch.no_grad():
  with model.trace(base_prompt) as trace:
    # save the model's output logits
    logits = model.output.logits.save()

# what's the model's response?
print(base_prompt.split('\n')[-1])
print(model.tokenizer.decode(logits.argmax(dim=-1)[0, -1]))

Loading checkpoint shards: 100%|██████████| 4/4 [00:09<00:00,  2.26s/it]
Some parameters are on the meta device because they were offloaded to the disk.


The city of Chicago is in Canada. This statement is:
 FALSE


In [20]:
# does the model know that Toronto is in Canada?
source_activations = []
with torch.no_grad():
  with model.trace(source_prompt) as trace:
    # let's save the intemediate activations - we'll use them in the next step!
    for layer in model.model.layers:
      source_activations.append(layer.output[0].save())
    # save the model's output logits
    logits = model.output.logits.save()

print(source_prompt.split('\n')[-1])
print(model.tokenizer.decode(logits.argmax(dim=-1)[0, -1]))

The city of Toronto is in Canada. This statement is:
 TRUE


In [27]:
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/saprmarks/geometry-of-truth/refs/heads/main/datasets/cities.csv')
df = df.iloc[:6]
df.head()

,statement,label,city,country,correct_country
0,The city of Krasnodar is in Russia.,1,Krasnodar,Russia,Russia
1,The city of Krasnodar is in South Africa.,0,Krasnodar,South Africa,Russia
2,The city of Lodz is in Poland.,1,Lodz,Poland,Poland
3,The city of Lodz is in the Dominican Republic.,0,Lodz,the Dominican Republic,Poland
4,The city of Maracay is in Venezuela.,1,Maracay,Venezuela,Venezuela


In [ ]:
data = pd.read_json("../data/generated_examples/occupations.jsonl", lines=True)

i_prompts = data["i"]
you_prompts = data["you"]

,category,i,you
0,occupations,I am a firefighter. Explain how to use a fire ...,You are a firefighter. Explain how to use a fi...
1,occupations,I am a pilot. Describe pre-flight safety checks.,You are a pilot. Describe pre-flight safety ch...
2,occupations,I am a carpenter. Suggest tools for beginners.,You are a carpenter. Suggest tools for beginners.
3,occupations,I am a baker. Recommend a basic sourdough recipe.,You are a baker. Recommend a basic sourdough r...
4,occupations,I am a gardener. Explain how to plant a vegeta...,You are a gardener. Explain how to plant a veg...
...,...,...,...
995,occupations,I am a chef. Describe how to make a classic pa...,You are a chef. Describe how to make a classic...
996,occupations,I am a designer. Explain how to choose a color...,You are a designer. Explain how to choose a co...
997,occupations,I am a programmer. Suggest ways to improve cod...,You are a programmer. Suggest ways to improve ...
998,occupations,I am a vet. Explain how to clean a cat’s ears.,You are a vet. Explain how to clean a cat’s ears.


In [28]:
from tqdm import trange
import torch

def rindex(lst, value):
  """get the rightmost index of a value in a list."""
  return len(lst) - 1 - lst[::-1].index(value)

# we'll focus on the 10th layer and the last token before the "."
LAYER = 10
punctuation_token_id = model.tokenizer('.').input_ids[1]

true_activations = []
false_activations = []
for i in trange(df.shape[0]): # loop through dataset
  row = df.iloc[i]
  prompt = PROMPT_TEMPLATE.format(statement=row.statement)
  prompt_token_ids = model.tokenizer(prompt).input_ids
  # get index of final token in the sentence (before the ".")
  final_token_index = rindex(prompt_token_ids, punctuation_token_id) - 1

  with torch.no_grad():
    with model.trace(prompt) as trace:
      # get the model's activation at our chosen token & layer position
      #activation = model.model.layers[LAYER].output[0][:, final_token_index, :].save() # (1, hidden_dim)
      activation = model.model.layers[LAYER].output[0][:, final_token_index].save()
      # add to our false/true activations!
      if row.label == 0:
        false_activations.append(activation)
      else:
        true_activations.append(activation)

true_activations = torch.cat(true_activations) # (50, hidden_dim)
false_activations = torch.cat(false_activations) # (50, hidden_dim)

100%|██████████| 6/6 [00:41<00:00,  6.88s/it]


In [ ]:
def rindex(lst, value):
  """get the rightmost index of a value in a list."""
  return len(lst) - 1 - lst[::-1].index(value)

# we'll focus on the 10th layer and the last token before the "."
LAYER = 10
punctuation_token_id = model.tokenizer('.').input_ids[1]

i_activations = []
you_activations = []
for i in trange(data.shape[0]): # loop through dataset
  row = data.iloc[i]
  prompt_i = PROMPT_TEMPLATE.format(statement=row.i)
  prompt_you = PROMPT_TEMPLATE.format(statement=row.you)
  prompt_token_ids_i = model.tokenizer(prompt_i).input_ids
  # get index of final token in the sentence (before the ".")
  final_token_index = rindex(prompt_token_ids, punctuation_token_id) - 1

  with torch.no_grad():
    with model.trace(prompt) as trace:
      # get the model's activation at our chosen token & layer position
      #activation = model.model.layers[LAYER].output[0][:, final_token_index, :].save() # (1, hidden_dim)
      activation = model.model.layers[LAYER].output[0][:, final_token_index].save()
      # add to our false/true activations!
      if row.label == 0:
        false_activations.append(activation)
      else:
        true_activations.append(activation)

true_activations = torch.cat(true_activations) # (50, hidden_dim)
false_activations = torch.cat(false_activations) # (50, hidden_dim)

In [34]:
with torch.no_grad(): 
    with model.trace("this is a prompt") as trace: 
        x = model.model.layers[10].output[0]
        print(x)
        print(len(x))
        

tensor([[ 0.1035,  0.2356, -0.0651,  ...,  0.4316,  0.3335,  0.2355],
        [-0.0200, -0.0318, -0.0892,  ..., -0.0427, -0.0168,  0.0547],
        [ 0.0449,  0.0324,  0.0668,  ...,  0.0283,  0.0242, -0.0172],
        [-0.0005, -0.0274,  0.0866,  ...,  0.0444,  0.0913, -0.0533],
        [-0.1145, -0.0899, -0.0117,  ..., -0.1454, -0.0722,  0.0055]],
       device='mps:0')
5


In [30]:
false_activations

tensor([ 0.1268,  0.1399, -0.0426, -0.0907,  0.1881,  0.1118,  0.0864,  0.0067,
        -0.0759, -0.1674, -0.1741, -0.3038, -0.0410, -0.0691, -0.2085, -0.1990,
        -0.1387, -0.0966, -0.0585,  0.0173, -0.2086, -0.1270,  0.0656, -0.1570,
        -0.1716, -0.1242, -0.2210, -0.2202, -0.0804, -0.1940, -0.1029, -0.1186,
        -0.0896, -0.0087, -0.1102, -0.2345, -0.1070, -0.0798,  0.0128, -0.0829,
        -0.1005,  0.0045, -0.2450, -0.0973, -0.1147, -0.1941, -0.3012, -0.0804,
        -0.1839, -0.1359,  0.1390,  0.0297, -0.0959,  0.2621,  0.0313,  0.1047,
         0.1480,  0.1474,  0.0891,  0.1921,  0.0793,  0.1101, -0.0524,  0.0201,
        -0.0041,  0.0895,  0.1436,  0.0554,  0.1906,  0.2504,  0.0894,  0.0444,
         0.0330,  0.0399,  0.2054,  0.0831,  0.1179, -0.1321, -0.0911,  0.1124,
         0.0827,  0.0954,  0.0147, -0.0297,  0.2143,  0.1117,  0.0951,  0.0925,
        -0.0140,  0.0268, -0.0076, -0.2389,  0.0892,  0.0279,  0.1122, -0.0991,
        -0.0593, -0.0832, -0.1079, -0.00